In [61]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine
import xgboost as xgb


In [62]:
# DB connection
engine = create_engine("postgresql+psycopg2://rirg2545@localhost/mimic")

In [63]:
# Mapping of drug ITEMIDs to labels and colors
ITEMID_LABELS = {
    221906: "Norepinephrine",
    221289: "Epinephrine",
    221662: "Dopamine",
    221986: "Milrinone",
    222315: "Vasopressin",
    221749: "Phenylephrine",
    221653: "Dobutamine",
    227692: "Isuprel"
}

ITEMID_COLORS = {
    221906: "blue",
    221289: "red",
    221662: "green",
    221986: "purple",
    222315: "orange",
    221749: "cyan",
    221653: "magenta",
    227692: "brown"
}

DATASETS = [
    "dobutamine", "dopamine", "epinephrine", "isuprel",
    "milrinone", "norepinephrine", "phenylephrine", "vasopressin"
]

# THRESHOLDS = {
#     221653: 0.000318,  # Dobutamine
#     221662: 0.001193,  # Dopamine
#     221289: 0.001011,  # Epinephrine
#     227692: 0.000004,  # Isuprel
#     221986: 0.000437,  # Milrinone
#     221906: 0.004640,  # Norepinephrine
#     221749: 0.006470,  # Phenylephrine
#     222315: 0.001269   # Vasopressin
#     mix: 
# }
THRESHOLDS = {
    "mix_80": 0.00115
}

MODELS_DIR = "../models/without_context_features"

In [64]:
# angepasst um treatment_count anzupassen
def preprocess_data(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Drop metadata/time columns and encode the label.
    """
    drop_cols = [
        "subject_id", "icustay_id",
        "context_start", "context_end",
        "target_start", "target_end", "treatment_given",
        "only_2_values"
    ]
    
    df_features = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # treatment_given already dropped ( treatment_count non-xistent)


    # Label as int
    df_features["label"] = df_features["positive_event"].astype(int)

    return df_features

def get_feature_cols(df_features: pd.DataFrame) -> list:
    """
    Identify feature columns (exclude label and split markers).
    """
    excluded = {"positive_event", "positive_sample", "split", "label"}
    feature_cols = [c for c in df_features.columns if c not in excluded]
    
    return feature_cols

def get_icustay_bounds(icustay_id):
    query = """
        SELECT intime, outtime
        FROM mimiciii.icustays
        WHERE icustay_id = %(icustay_id)s;
    """
    df = pd.read_sql(query, engine, params={"icustay_id": icustay_id})
    if df.empty:
        raise ValueError("ICU stay not found")
    return df.iloc[0]['intime'], df.iloc[0]['outtime']

def load_ground_truth(icustay_id):  
    
    df = pd.read_sql(f"""
        SELECT target_start, target_end
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s AND positive_event IS TRUE
    """, engine, params={"icustay_id": icustay_id})
    
    return df

def load_predictions(icustay_id):
    # 1) Load data (same table used for training)
    df = pd.read_sql(
        """
        SELECT *
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
          AND split = 'test'
        ORDER BY context_end
        """,
        engine,
        params={"icustay_id": icustay_id},
    )

    if df.empty:
        return pd.DataFrame(
            columns=["target_start", "target_end", "pred_proba", "drug_label", "color"]
        )

    # 2) Datetime parsing (for plotting later)
    for col in ["context_start", "context_end", "target_start", "target_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 3) Reproduce training preprocessing
    df_features = preprocess_data(
        df)

    feature_cols = get_feature_cols(df_features)

    X = df_features[feature_cols]
    y = df_features["label"]  # not strictly needed for plotting

    # 4) DMatrix (exactly like training)
    dmat = xgb.DMatrix(X, missing=np.nan)

    # 5) Load trained Booster
    model_path = "/dss/work/rirg2545/actionable-hypotension/models_given/optimized/xgb_mix.json"
    bst = xgb.Booster()
    bst.load_model(model_path)

    # 6) Predict
    preds = bst.predict(dmat)
    df = df.assign(pred_proba=preds)

    # 7) Thresholding (single model → single threshold)
    threshold = THRESHOLDS.get("mix_80", 0.01)
    pred_df = df[df["pred_proba"] >= threshold].copy()

    pred_df["drug_label"] = "Hypotension Risk"
    pred_df["color"] = "red"  # z.B. Signalrot für Warnung

    return pred_df[
        ["target_start", "target_end", "pred_proba", "drug_label", "color"]
    ]


def plot_blocks(fig, df, row):
    for entry in df.itertuples():
        fig.add_trace(go.Scatter(
            x=[entry.target_start, entry.target_end],
            y=[1, 1],
            mode="lines",
            line=dict(color="red", width=10),
            name="Target Window",
            showlegend=False,
            opacity=0.3,
            hovertemplate=f"Target Window<br>%{{x}}"
        ), row=row, col=1)
    fig.update_yaxes(title="Target Windows", row=row)

In [65]:
def load_patient_metadata(icustay_id):
    
    df = pd.read_sql(f"""
        SELECT gender, ethnicity_group, age, height, weight, obesity, hypertension, diabetes, kidney_disease, lung_disease, heart_disease, 
        FROM ce_approach.mv_metadata
        WHERE icustay_id = %(icustay_id)s
    """, engine, params={"icustay_id": icustay_id})

    return df

In [76]:
def load_medications(icustay_id):
    df = pd.read_sql(
        """
        SELECT sedatives_given, blood_products_transfusions_given, antibiotics_given, anticoagulants_antiplatelets_given, 
        neuromuscular_blockers_given, analgesics_given, crystalloids_given, electrolytes_given, gi_protection_given,
        parenteral_nutrition_given, antiarrhythmics_given, context_end
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
          AND split = 'test'
        ORDER BY context_end
        """,
        engine,
        params={"icustay_id": icustay_id},
    )
    return df

In [67]:
def load_available_context_windows(icustay_id):
    
    df = pd.read_sql(f"""
        SELECT context_start, context_end
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
    """, engine, params={"icustay_id": icustay_id})

    return df

def plot_context_windows(fig, df, row):
    # Da keine itemid mehr existiert, alles in einer Zeile plotten (y=1)
    for entry in df.itertuples():
        fig.add_trace(go.Scatter(
            x=[entry.context_start, entry.context_end],
            y=[1, 1],  # Nur eine Zeile für alle Kontextfenster
            mode="lines",
            line=dict(color="gray", width=10),
            name="Context Window",
            showlegend=False,
            opacity=0.3,
            hovertemplate=f"Context Window<br>%{{x}}"
        ), row=row, col=1)

    fig.update_yaxes(title="Context Windows", row=row, range=[0.5, 1.5], showticklabels=False)

In [68]:
def load_treatments(icustay_id):
    query = """
        SELECT treatment_starttime, treatment_endtime, itemid
        FROM ce_approach.linkorder_treatment_events
        WHERE icustay_id = %(icustay_id)s;
    """
    df = pd.read_sql(query, engine, params={"icustay_id": icustay_id})
    df["drug_label"] = df["itemid"].map(ITEMID_LABELS).fillna(df["itemid"].astype(str))
    df["color"] = df["itemid"].map(ITEMID_COLORS).fillna("gray")
    return df

def plot_treatments(fig, df, row):
    row_offset_map = {itemid: i for i, itemid in enumerate(sorted(df["itemid"].unique()))}
    for entry in df.itertuples():
        y = row_offset_map[entry.itemid] + 1
        fig.add_trace(go.Scatter(
            x=[entry.treatment_starttime, entry.treatment_endtime],
            y=[y, y],
            mode="lines",
            line=dict(color=entry.color, width=10),
            name=entry.drug_label,
            showlegend=False,
            hovertemplate=f"{entry.drug_label}<br>%{{x}}"
        ), row=row, col=1)
    fig.update_yaxes(title="Treatment", row=row)

In [69]:
def load_map_from_mix_windows(icustay_id):
    df = pd.read_sql(
        """
        SELECT
            context_start,
            context_end,
            map_values_filtered
        FROM ce_approach.mix_windows
        WHERE icustay_id = %(icustay_id)s
        ORDER BY context_start
        """,
        engine,
        params={"icustay_id": icustay_id},
    )

    rows = []
    for _, r in df.iterrows():
        for e in r.map_values_filtered:
            charttime = r.context_start + pd.to_timedelta(e["pos"], unit="s")
            valuenum = e["value"]
            rows.append(
                {
                    "charttime": charttime,
                    "valuenum": valuenum
                }
            )

    return pd.DataFrame(rows)

def plot_map_values(fig, df, row):
    fig.add_trace(go.Scatter(
        x=df["charttime"],
        y=df["valuenum"],
        mode="markers+lines",
        line=dict(width=1),
        marker=dict(size=4),
        name="MAP (mmHg)",
        marker_color="black",
        showlegend=False,
        hovertemplate="MAP @ %{x}: %{y} mmHg"
    ), row=row, col=1)
    fig.update_yaxes(title="MAP (mmHg)", row=row)

In [70]:
def plot_positive_and_predicted_windows(icustay_id):
    treat = load_treatments(icustay_id) # treatment_events that actually happened
    ctx = load_available_context_windows(icustay_id) #context_start, context_end
    map_df = load_map_from_mix_windows(icustay_id) # load map values the model saw
    pos = load_ground_truth(icustay_id)
    pred = load_predictions(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id) # get intime outtime

    fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=(
                                        "Drug Treatments",
                                        "Available Context Windows",
                                        "MAP Values",
                                        "Positive Events (Ground Truth)", 
                                        "Model Predictions" 
                                        ))

    plot_treatments(fig, treat, row=1)
    plot_context_windows(fig, ctx, row=2)
    plot_map_values(fig, map_df, row=3)
    plot_blocks(fig, pos, row=4)
    plot_blocks(fig, pred, row=5)

    fig.update_layout(
        height=1000,
        title=f"Treatment Windows for ICU Stay {icustay_id}",
        xaxis=dict(range=[intime, outtime]),
        xaxis2=dict(range=[intime, outtime]),
        xaxis3=dict(range=[intime, outtime]),
        xaxis4=dict(range=[intime, outtime]),
        xaxis5=dict(range=[intime, outtime])
    )
    
    #fig.write_html(f"../plots/{icustay_id}.html")
    fig.show()

In [71]:
#plot_positive_and_predicted_windows(icustay_id=200349)

In [72]:
#plot_positive_and_predicted_windows(icustay_id=200349)

In [73]:
## GIF production
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from datetime import timedelta
import imageio.v2 as imageio
from pathlib import Path
import tempfile

def create_prediction_gif(icustay_id, interval_minutes=15, frames_around_event=10, output_path=None):
    """
    Create a GIF showing how model predictions evolve over time, focused on ground truth events.
    
    Parameters:
    -----------
    icustay_id : int
        The ICU stay identifier
    interval_minutes : int
        Time interval between frames in minutes (default: 15)
    frames_around_event : int
        Number of frames to show before and after each ground truth event (default: 10)
    output_path : str or None
        Path to save the GIF. If None, saves to f"../plots/{icustay_id}_predictions.gif"
    """
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_timeline_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, frame_path
        )
        frame_paths.append(str(frame_path))
    
    # Create GIF
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions.gif"
    
    print(f"Creating GIF at {output_path}...")
    create_gif_from_frames(frame_paths, output_path, duration=0.5)
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"GIF created successfully: {output_path}")

def create_gif_from_frames(frame_paths, output_path, duration=0.5):
    """Create a GIF from a list of image paths."""
    images = []
    for path in frame_paths:
        images.append(imageio.imread(path))
    
    imageio.mimsave(output_path, images, duration=duration, loop=0)
    
def create_prediction_video(icustay_id, interval_minutes=15, frames_around_event=10, output_path=None, fps=5):
    """Create an MP4 video instead of GIF"""
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_timeline_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, frame_path
        )
        frame_paths.append(str(frame_path))
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions.mp4"
    
    # Create video with imageio
    writer = imageio.get_writer(output_path, fps=fps, codec='libx264', quality=8)
    
    for frame_path in frame_paths:
        frame = imageio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()


def create_timeline_frame(icustay_id, treat, map_df, pos, pred, 
                          intime, outtime, current_time, save_path):
    """
    Create a simple timeline visualization showing:
    1. MAP values over time (context)
    2. Ground truth positive events
    3. Model predictions
    4. Treatment starts
    """
    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f'ICU Stay - Time: {current_time.strftime("%Y-%m-%d %H:%M")}', 
                 fontsize=16, fontweight='bold')
    
    # Convert times to hours from admission for easier plotting
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # ============== Panel 1: MAP Values ==============
    ax1 = axes[0]
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        ax1.plot(hours, map_df['valuenum'], 'k-', linewidth=1, marker='o', markersize=3)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.3, label='Hypotension Threshold (65mmHg)')
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='Current time')
    ax1.set_ylabel('MAP (mmHg)', fontsize=12, fontweight='bold')
    ax1.set_title('Context: Mean Arterial Pressure', fontsize=11, loc='left')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    ax1.set_ylim(40, 120)
    
    # ============== Panel 2: Ground Truth Positive Events ==============
    ax2 = axes[1]
    ax2.set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
    ax2.set_title('True Catecholamine Initiation', fontsize=11, loc='left')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        width = end_hour - start_hour
        
        # Only show if event has started by current time
        if start_hour <= current_hour:
            # Color based on whether event is in past or ongoing
            if end_hour <= current_hour:
                color = 'darkred'
                alpha = 0.5
            else:
                color = 'red'
                alpha = 0.8
            
            rect = Rectangle((start_hour, 0.5), width, 1.0, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=1)
            ax2.add_patch(rect)
    
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # ============== Panel 3: Model Predictions ==============
    ax3 = axes[2]
    ax3.set_ylabel('Predictions', fontsize=12, fontweight='bold')
    ax3.set_title('Model Predicted Risk Windows', fontsize=11, loc='left')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            width = end_hour - start_hour
            
            # Predictions fade based on when they were made
            pred_time = to_hours(p['context_end'])
            hours_old = current_hour - pred_time
            alpha = max(0.3, 1.0 - (hours_old / 24))  # Fade over 24 hours
            
            rect = Rectangle((start_hour, 0.5), width, 1.0,
                           facecolor='orange', edgecolor='black', alpha=alpha, linewidth=1)
            ax3.add_patch(rect)
    
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax3.grid(True, alpha=0.3, axis='x')
    
    # ============== Panel 4: Treatment Starts ==============
    ax4 = axes[3]
    ax4.set_ylabel('Treatments', fontsize=12, fontweight='bold')
    ax4.set_title('Vasopressor Treatment Initiations', fontsize=11, loc='left')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    ax4.set_xlabel('Hours from ICU Admission', fontsize=12, fontweight='bold')
    
    # Show treatment starts as vertical lines
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        ax4.axvline(x=start_hour, color='green', linewidth=3, alpha=0.7)
        ax4.text(start_hour, 1.5, t['drug_label'], rotation=90, 
                va='bottom', ha='right', fontsize=8)
    
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax4.grid(True, alpha=0.3, axis='x')
    
    # Set x-axis limits for all panels
    for ax in axes:
        ax.set_xlim(0, total_hours)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()

def generate_focused_time_points(pos_df, interval_minutes, frames_around, intime, outtime):
    """
    Generate time points focused around ground truth events.
    """
    time_points_set = set()
    interval_delta = timedelta(minutes=interval_minutes)
    
    for _, event in pos_df.iterrows():
        # Find center of ground truth event
        event_center = event['target_start'] + (event['target_end'] - event['target_start']) / 2
        
        # Generate frames around this event
        for offset in range(-frames_around, frames_around + 1):
            time_point = event_center + (offset * interval_delta)
            
            # Keep within ICU stay bounds
            if intime <= time_point <= outtime:
                time_points_set.add(time_point)
    
    # Sort time points
    time_points = sorted(list(time_points_set))
    
    return time_points

def load_all_predictions_data(icustay_id):
    """Load all prediction data without filtering by time."""
    df = pd.read_sql(
        """
        SELECT *
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
          AND split = 'test'
        ORDER BY context_end
        """,
        engine,
        params={"icustay_id": icustay_id},
    )
    
    if df.empty:
        return pd.DataFrame()
    
    # Parse datetime columns
    for col in ["context_start", "context_end", "target_start", "target_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    # Preprocess and predict
    df_features = preprocess_data(df)
    feature_cols = get_feature_cols(df_features)
    X = df_features[feature_cols]
    
    dmat = xgb.DMatrix(X, missing=np.nan)
    
    # Load model
    model_path = "/dss/work/rirg2545/actionable-hypotension/models_given/optimized/xgb_mix.json"
    bst = xgb.Booster()
    bst.load_model(model_path)
    
    # Predict
    preds = bst.predict(dmat)
    df = df.assign(pred_proba=preds)
    
    # Apply threshold
    threshold = THRESHOLDS.get("mix_80", 0.01)
    df["predicted_positive"] = df["pred_proba"] >= threshold
    
    return df

def filter_predictions_by_time(df, current_time):
    """Filter predictions to only show those available at current_time."""
    if df.empty:
        return pd.DataFrame(
            columns=["target_start", "target_end", "pred_proba", "context_end"]
        )
    
    # Only show predictions where the context_end is <= current_time
    pred_df = df[
        (df["context_end"] <= current_time) & 
        (df["predicted_positive"] == True)
    ].copy()
    
    return pred_df[
        ["target_start", "target_end", "pred_proba", "context_end"]
    ]







In [ ]:
# Mit Medikamente

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from datetime import timedelta
import imageio
from pathlib import Path
import tempfile

def create_zoomed_prediction_video(icustay_id, interval_minutes=15, frames_around_event=10, 
                                   window_hours=6, output_path=None, fps=5):
    """
    Create an MP4 video with a zoomed-in rolling window view.
    Shows a fixed time window (e.g., 6 hours) that moves with the current time.
    
    Parameters:
    -----------
    icustay_id : int
        The ICU stay identifier
    interval_minutes : int
        Time interval between frames in minutes (default: 15)
    frames_around_event : int
        Number of frames to show before and after each ground truth event (default: 10)
    window_hours : float
        Size of the rolling window in hours (default: 6)
    output_path : str or None
        Path to save the video
    fps : int
        Frames per second for the video (default: 5)
    """
    import imageio.v2 as iio  # Use v2 API explicitly
    # Load all data
    treat = load_treatments(icustay_id)
    meds = load_medications(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    metadata = load_patient_metadata(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating zoomed frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create zoomed plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_zoomed_timeline_frame(
            icustay_id, treat, meds, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, window_hours, metadata, frame_path
        )
        frame_paths.append(str(frame_path))
    
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions_meds_zoomed.mp4"
    
    # Create video with imageio (v2 API, fixed dimensions)
    writer = iio.get_writer(output_path, fps=fps, codec='libx264', quality=8, 
                           macro_block_size=1)
    
    for frame_path in frame_paths:
        frame = iio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"Zoomed video created successfully: {output_path}")


def create_zoomed_timeline_frame(icustay_id, treat, meds, map_df, pos, pred, 
                                 intime, outtime, current_time, window_hours, metadata, save_path):
    """
    Create a zoomed-in timeline visualization with a rolling window.
    
    Parameters:
    -----------
    window_hours : float
        Size of the window in hours (e.g., 6 shows ±3 hours from current time)
    metadata : pd.DataFrame
        Patient metadata including demographics and comorbidities
    """
    fig = plt.figure(figsize=(20, 11), constrained_layout=True)
    
    # Create grid: 1 row for metadata panel, 4 rows for timeline
    #gs = fig.add_gridspec(5, 1, height_ratios=[0.8, 1, 1, 1, 1], hspace=0.15)
    gs = fig.add_gridspec(
    5, 1,
    height_ratios=[1.3, 1, 1, 1, 1]
)
    # Metadata panel at top
    ax_meta = fig.add_subplot(gs[0, 0])
    
    # Timeline panels
    axes = [fig.add_subplot(gs[i+1, 0]) for i in range(4)]
    
    # Add patient metadata panel
    add_patient_metadata_panel(ax_meta, metadata, icustay_id)
    
    fig.suptitle(f'ICU Stay {icustay_id} - Time: {current_time.strftime("%Y-%m-%d %H:%M")} [Zoomed View: ±{window_hours/2:.1f}h]', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Convert times to hours from admission for easier plotting
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # Calculate window bounds
    window_start = max(0, current_hour - window_hours / 2)
    window_end = min(total_hours, current_hour + window_hours / 2)
    
    # Adjust if we're at the boundaries
    if current_hour < window_hours / 2:
        window_end = min(total_hours, window_hours)
    if current_hour > total_hours - window_hours / 2:
        window_start = max(0, total_hours - window_hours)
    
    # ============== Panel 1: MAP Values ==============
    ax1 = axes[0]
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        values = map_df['valuenum'].values
        
        # Filter to window
        window_mask = [(h >= window_start and h <= window_end) for h in hours]
        window_hours_list = [h for h, m in zip(hours, window_mask) if m]
        window_values = [v for v, m in zip(values, window_mask) if m]
        
        if window_hours_list:
            ax1.plot(window_hours_list, window_values, 'k-', linewidth=2, marker='o', markersize=5)
        
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Hypotension Threshold (65mmHg)')
    
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8, label='Current time')
    ax1.set_ylabel('MAP (mmHg)', fontsize=14, fontweight='bold')
    ax1.set_title('Context: Mean Arterial Pressure', fontsize=12, loc='left', fontweight='bold')
    ax1.grid(True, alpha=0.4, linewidth=0.5)
    ax1.legend(loc='upper right', fontsize=11)
    ax1.set_ylim(40, 120)
    
    # ============== Panel 2: Ground Truth Positive Events ==============
    ax2 = axes[1]
    ax2.set_ylabel('Ground Truth', fontsize=14, fontweight='bold')
    ax2.set_title('True Catecholamine Initiation Windows', fontsize=12, loc='left', fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        
        # Only show if visible in window and has started
        if start_hour <= current_hour and end_hour >= window_start and start_hour <= window_end:
            width = end_hour - start_hour
            
            # Color based on whether event is in past or ongoing
            if end_hour <= current_hour:
                color = 'darkred'
                alpha = 0.6
                label = 'Past Event'
            else:
                color = 'red'
                alpha = 0.9
                label = 'Ongoing Event'
            
            rect = Rectangle((start_hour, 0.3), width, 1.4, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=2)
            ax2.add_patch(rect)
            
            # Add label at start of event
            if start_hour >= window_start:
                ax2.text(start_hour, 1.0, '▼', ha='center', va='center', 
                        fontsize=16, fontweight='bold', color='darkred')
    
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax2.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # ============== Panel 3: Model Predictions ==============
    ax3 = axes[2]
    ax3.set_ylabel('Predictions', fontsize=14, fontweight='bold')
    ax3.set_title('Model Predicted Risk Windows', fontsize=12, loc='left', fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            
            # Only show if visible in window
            if end_hour >= window_start and start_hour <= window_end:
                width = end_hour - start_hour
                
                # Predictions fade based on when they were made
                pred_time = to_hours(p['context_end'])
                hours_old = current_hour - pred_time
                alpha = max(0.4, 1.0 - (hours_old / 12))  # Fade over 12 hours
                
                rect = Rectangle((start_hour, 0.3), width, 1.4,
                               facecolor='orange', edgecolor='black', alpha=alpha, linewidth=2)
                ax3.add_patch(rect)
                
                # Add marker at prediction time
                if pred_time >= window_start and pred_time <= window_end:
                    ax3.plot(pred_time, 1.0, 'o', color='darkorange', markersize=8, 
                            markeredgecolor='black', markeredgewidth=1.5, zorder=10)
    
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax3.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # ============== Panel 4: Treatment Starts ==============
    MEDICATION_STYLE = {
        "antibiotics_given": dict(color="tab:blue", label="Antibiotics"),
        "sedatives_given": dict(color="tab:purple", label="Sedatives"),
        "analgesics_given": dict(color="saddlebrown", label="Analgesics"),
        "crystalloids_given": dict(color="tab:orange", label="Fluids"),
        "anticoagulants_antiplatelets_given": dict(color="tab:gray", label="Anticoagulants"),
    }

    ax4 = axes[3]
    ax4.set_ylabel('Treatments', fontsize=14, fontweight='bold')
    ax4.set_title('Vasopressor Treatment Initiations', fontsize=12, loc='left', fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    ax4.set_xlabel('Hours from ICU Admission', fontsize=14, fontweight='bold')
    
    # Show treatment starts as vertical lines
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        
        # Only show if in window
        if start_hour >= window_start and start_hour <= window_end:
            ax4.axvline(x=start_hour, color='green', linewidth=4, alpha=0.8, zorder=5)
            ax4.plot(start_hour, 1.0, 'v', color='darkgreen', markersize=12, 
                    markeredgecolor='black', markeredgewidth=1.5, zorder=10)
            ax4.text(start_hour, 0.3, t['drug_label'], rotation=45, 
                    va='top', ha='right', fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))
    
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax4.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # Set x-axis limits for all panels (zoomed window)
    for ax in axes:
        ax.set_xlim(window_start, window_end)
        # Add time markers every hour
        ax.set_xticks(np.arange(np.ceil(window_start), np.floor(window_end) + 1, 1))
    
    # Add minor gridlines every 15 minutes
    for ax in axes:
        ax.set_xticks(np.arange(window_start, window_end, 0.25), minor=True)
        ax.grid(True, which='minor', alpha=0.2, linewidth=0.3)

    handles = [
        plt.Line2D([0], [0], color=v["color"], marker="|",
                   linestyle="None", markersize=12, label=v["label"])
        for v in MEDICATION_STYLE.values()
    ]

    ax4.legend(
        handles=handles,
        loc="upper left",
        fontsize=9,
        framealpha=0.9,
        
    )

    # ============== Medication Context (non-vasopressor) ==============
    # meds: eine Zeile pro Kontextfenster, mit context_end

    meds_visible = meds[
        meds["context_end"] <= current_time
    ]

    for _, row in meds_visible.iterrows():
        hour = to_hours(row["context_end"])

        if hour < window_start or hour > window_end:
            continue

        for col, style in MEDICATION_STYLE.items():
            if not bool(row[col]):
                continue

            ax4.plot(
                hour,
                0.6,
                marker="|",
                color=style["color"],
                markersize=18,
                markeredgewidth=3,
                alpha=0.9,
                zorder=6,
            )
    
    #plt.tight_layout(rect=[0, 0, 1, 0.99])  # Leave space for suptitle
    # Use fixed bbox instead of 'tight' to ensure consistent dimensions
    plt.savefig(save_path, dpi=120, bbox_inches=None)
    plt.close()


def add_patient_metadata_panel(ax, metadata, icustay_id):
    """
    Add a compact patient metadata panel showing demographics and comorbidities.
    Uses color-coded boxes for comorbidities (present/absent).
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    if metadata.empty:
        ax.text(0.5, 0.5, 'No metadata available', ha='center', va='center', fontsize=12)
        return
    
    patient = metadata.iloc[0]
    
    # Background box
    bg_rect = Rectangle((0.005, 0.05), 0.99, 0.9, facecolor='#f8f9fa', 
                        edgecolor='#dee2e6', linewidth=2, zorder=0)
    ax.add_patch(bg_rect)
    
    # Left side: Demographics
    y_pos = 0.75
    demographics_x = 0.02
    
    # Format demographics
    gender = patient.get('gender', 'Unknown')
    ethnicity = patient.get('ethnicity_group', 'Unknown')
    age = patient.get('age', 'Unknown')
    height = patient.get('height', None)
    weight = patient.get('weight', None)
    
    # Calculate BMI if available
    bmi_text = ""
    if height and weight and height > 0:
        bmi = weight / ((height / 100) ** 2)
        bmi_text = f" (BMI: {bmi:.1f})"
    
    demo_text = (
        f"Gender: {gender}  |  Age: {age} yrs  |  Ethnicity: {ethnicity}\n"
        f"Height: {height if height else 'N/A'} cm  |  Weight: {weight if weight else 'N/A'} kg{bmi_text}"
    )
    
    ax.text(demographics_x, y_pos, demo_text, fontsize=10, va='top', 
            fontweight='normal', family='monospace')
    
    # Right side: Comorbidities with color-coded indicators
    comorb_x_start = 0.02
    comorb_y = 0.35
    
    ax.text(comorb_x_start, comorb_y + 0.12, 'Comorbidities:', 
            fontsize=11, fontweight='bold', va='top')
    
    # Define comorbidities to display
    comorbidities = [
        ('obesity', 'Obesity'),
        ('hypertension', 'Hypertension'),
        ('diabetes', 'Diabetes'),
        ('kidney_disease', 'Kidney Disease'),
        ('lung_disease', 'Lung Disease'),
        ('heart_disease', 'Heart Disease'),
    ]
    
    # Create color-coded boxes for each comorbidity
    box_width = 0.15
    box_height = 0.08
    boxes_per_row = 3
    x_spacing = 0.16
    y_spacing = 0.11
    
    for idx, (col_name, display_name) in enumerate(comorbidities):
        row = idx // boxes_per_row
        col = idx % boxes_per_row
        
        box_x = comorb_x_start + col * x_spacing
        box_y = comorb_y - row * y_spacing
        
        # Check if comorbidity is present
        is_present = patient.get(col_name, 0) == 1
        
        # Color scheme
        if is_present:
            box_color = '#ffcccc'  # Light red
            edge_color = '#cc0000'  # Dark red
            text_color = '#cc0000'
            text_weight = 'bold'
            symbol = '✓'
        else:
            box_color = '#e8f4f8'  # Light blue-gray
            edge_color = '#b0bec5'  # Gray
            text_color = '#666666'
            text_weight = 'normal'
            symbol = '−'
        
        # Draw box
        rect = Rectangle((box_x, box_y - box_height), box_width, box_height,
                        facecolor=box_color, edgecolor=edge_color, 
                        linewidth=2, zorder=1)
        ax.add_patch(rect)
        
        # Add text
        ax.text(box_x + box_width/2, box_y - box_height/2, 
               f"{symbol} {display_name}", 
               ha='center', va='center', fontsize=9, 
               color=text_color, fontweight=text_weight, zorder=2)
    
    # Add legend/key at bottom right
    legend_x = 0.85
    legend_y = 0.15
    
    # Present indicator
    small_box = Rectangle((legend_x, legend_y), 0.02, 0.05, 
                          facecolor='#ffcccc', edgecolor='#cc0000', linewidth=1.5)
    ax.add_patch(small_box)
    ax.text(legend_x + 0.025, legend_y + 0.025, 'Present', 
           va='center', fontsize=8, color='#666666')
    
    # Absent indicator
    small_box2 = Rectangle((legend_x, legend_y - 0.08), 0.02, 0.05, 
                           facecolor='#e8f4f8', edgecolor='#b0bec5', linewidth=1.5)
    ax.add_patch(small_box2)
    ax.text(legend_x + 0.025, legend_y - 0.055, 'Absent', 
           va='center', fontsize=8, color='#666666')


def create_dual_view_video(icustay_id, interval_minutes=15, frames_around_event=10, 
                           window_hours=6, output_path=None, fps=5):
    """
    Create a video with both full timeline and zoomed view side by side.
    
    Top: Full timeline overview
    Bottom: Zoomed rolling window
    """
    import imageio.v2 as iio  # Use v2 API explicitly
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty or pos.empty:
        print(f"Insufficient data for ICU stay {icustay_id}")
        return
    
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames with dual view")
    
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}")
        
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_dual_view_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, window_hours, frame_path
        )
        frame_paths.append(str(frame_path))
    
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions_dual.mp4"
    
    writer = iio.get_writer(output_path, fps=fps, codec='libx264', quality=8,
                           macro_block_size=1)
    
    for frame_path in frame_paths:
        frame = iio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"Dual-view video created: {output_path}")


def create_dual_view_frame(icustay_id, treat, map_df, pos, pred, 
                           intime, outtime, current_time, window_hours, save_path):
    """Create a frame with both overview and zoomed view."""
    
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(8, 1, hspace=0.3)
    
    # Top section: Full overview (4 rows)
    ax_overview = [fig.add_subplot(gs[i, 0]) for i in range(4)]
    
    # Bottom section: Zoomed view (4 rows)
    ax_zoom = [fig.add_subplot(gs[i+4, 0]) for i in range(4)]
    
    fig.suptitle(f'ICU Stay {icustay_id} - Time: {current_time.strftime("%Y-%m-%d %H:%M")}', 
                 fontsize=18, fontweight='bold')
    
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # Plot full overview
    plot_overview_section(ax_overview, map_df, pos, pred, treat, intime, current_time, 
                         current_hour, total_hours, to_hours, "OVERVIEW")
    
    # Plot zoomed section
    plot_zoomed_section(ax_zoom, map_df, pos, pred, treat, intime, outtime, current_time, 
                       current_hour, total_hours, window_hours, to_hours, "ZOOMED VIEW")
    
    plt.savefig(save_path, dpi=100, bbox_inches=None)
    plt.close()


def plot_overview_section(axes, map_df, pos, pred, treat, intime, current_time, 
                          current_hour, total_hours, to_hours, title_prefix):
    """Helper to plot full overview."""
    # Similar to create_timeline_frame but with title prefix
    ax1, ax2, ax3, ax4 = axes
    
    # MAP
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        ax1.plot(hours, map_df['valuenum'], 'k-', linewidth=0.5, marker='o', markersize=2)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.3)
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax1.set_ylabel('MAP', fontsize=10, fontweight='bold')
    ax1.set_title(f'{title_prefix} - Mean Arterial Pressure', fontsize=10, loc='left')
    ax1.set_ylim(40, 120)
    ax1.grid(True, alpha=0.2)
    
    # Ground truth
    ax2.set_ylabel('Truth', fontsize=10, fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        if start_hour <= current_hour:
            color = 'darkred' if end_hour <= current_hour else 'red'
            alpha = 0.5 if end_hour <= current_hour else 0.8
            rect = Rectangle((start_hour, 0.5), end_hour - start_hour, 1.0, 
                           facecolor=color, alpha=alpha)
            ax2.add_patch(rect)
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax2.grid(True, alpha=0.2, axis='x')
    
    # Predictions
    ax3.set_ylabel('Pred', fontsize=10, fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            pred_time = to_hours(p['context_end'])
            hours_old = current_hour - pred_time
            alpha = max(0.3, 1.0 - (hours_old / 24))
            rect = Rectangle((start_hour, 0.5), end_hour - start_hour, 1.0,
                           facecolor='orange', alpha=alpha)
            ax3.add_patch(rect)
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax3.grid(True, alpha=0.2, axis='x')
    
    # Treatments
    ax4.set_ylabel('Treat', fontsize=10, fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        ax4.axvline(x=start_hour, color='green', linewidth=2, alpha=0.7)
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax4.grid(True, alpha=0.2, axis='x')
    ax4.set_xlabel('Hours from ICU Admission', fontsize=10)
    
    for ax in axes:
        ax.set_xlim(0, total_hours)


def plot_zoomed_section(axes, map_df, pos, pred, treat, intime, outtime, current_time, 
                       current_hour, total_hours, window_hours, to_hours, title_prefix):
    """Helper to plot zoomed view."""
    # Calculate window
    window_start = max(0, current_hour - window_hours / 2)
    window_end = min(total_hours, current_hour + window_hours / 2)
    
    if current_hour < window_hours / 2:
        window_end = min(total_hours, window_hours)
    if current_hour > total_hours - window_hours / 2:
        window_start = max(0, total_hours - window_hours)
    
    ax1, ax2, ax3, ax4 = axes
    
    # MAP (zoomed)
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        values = map_df['valuenum'].values
        window_mask = [(h >= window_start and h <= window_end) for h in hours]
        window_hours_list = [h for h, m in zip(hours, window_mask) if m]
        window_values = [v for v, m in zip(values, window_mask) if m]
        if window_hours_list:
            ax1.plot(window_hours_list, window_values, 'k-', linewidth=2, marker='o', markersize=5)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.5, linewidth=2)
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax1.set_ylabel('MAP', fontsize=12, fontweight='bold')
    ax1.set_title(f'{title_prefix} (±{window_hours/2:.1f}h) - MAP', fontsize=11, loc='left', fontweight='bold')
    ax1.set_ylim(40, 120)
    ax1.grid(True, alpha=0.4)
    
    # Ground truth (zoomed)
    ax2.set_ylabel('Truth', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        if start_hour <= current_hour and end_hour >= window_start and start_hour <= window_end:
            color = 'darkred' if end_hour <= current_hour else 'red'
            alpha = 0.6 if end_hour <= current_hour else 0.9
            rect = Rectangle((start_hour, 0.3), end_hour - start_hour, 1.4, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=2)
            ax2.add_patch(rect)
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax2.grid(True, alpha=0.4, axis='x')
    
    # Predictions (zoomed)
    ax3.set_ylabel('Pred', fontsize=12, fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            if end_hour >= window_start and start_hour <= window_end:
                pred_time = to_hours(p['context_end'])
                hours_old = current_hour - pred_time
                alpha = max(0.4, 1.0 - (hours_old / 12))
                rect = Rectangle((start_hour, 0.3), end_hour - start_hour, 1.4,
                               facecolor='orange', edgecolor='black', alpha=alpha, linewidth=2)
                ax3.add_patch(rect)
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax3.grid(True, alpha=0.4, axis='x')
    
    # Treatments (zoomed)
    ax4.set_ylabel('Treat', fontsize=12, fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        if start_hour >= window_start and start_hour <= window_end:
            ax4.axvline(x=start_hour, color='green', linewidth=4, alpha=0.8)
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax4.grid(True, alpha=0.4, axis='x')
    ax4.set_xlabel('Hours from ICU Admission', fontsize=12, fontweight='bold')
    
    for ax in axes:
        ax.set_xlim(window_start, window_end)

def load_patient_metadata(icustay_id):
    """Load patient demographics and comorbidities."""
    df = pd.read_sql("""
        SELECT gender, ethnicity_group, age, height, weight, 
               obesity, hypertension, diabetes, kidney_disease, 
               lung_disease, heart_disease
        FROM ce_approach.mv_metadata
        WHERE icustay_id = %(icustay_id)s
    """, engine, params={"icustay_id": icustay_id})
    return df


# Example usage:
create_zoomed_prediction_video(icustay_id=202300, interval_minutes=15, frames_around_event=20, window_hours=8)
# create_dual_view_video(icustay_id=229240, interval_minutes=15, frames_around_event=10, window_hours=6)

Generated 29 frames focused on 1 ground truth events
Time range: 2192-06-20 18:14:22 to 2192-06-21 01:14:22
Generating zoomed frames...
Frame 1/29: 2192-06-20 18:14:22
Frame 11/29: 2192-06-20 20:44:22
Frame 21/29: 2192-06-20 23:14:22
Zoomed video created successfully: /dss/work/rirg2545/actionable-hypotension/simulation/202300_predictions_meds_zoomed.mp4


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from datetime import timedelta
import imageio
from pathlib import Path
import tempfile

def create_zoomed_prediction_video(icustay_id, interval_minutes=15, frames_around_event=10, 
                                   window_hours=6, output_path=None, fps=5):
    """
    Create an MP4 video with a zoomed-in rolling window view.
    Shows a fixed time window (e.g., 6 hours) that moves with the current time.
    
    Parameters:
    -----------
    icustay_id : int
        The ICU stay identifier
    interval_minutes : int
        Time interval between frames in minutes (default: 15)
    frames_around_event : int
        Number of frames to show before and after each ground truth event (default: 10)
    window_hours : float
        Size of the rolling window in hours (default: 6)
    output_path : str or None
        Path to save the video
    fps : int
        Frames per second for the video (default: 5)
    """
    import imageio.v2 as iio  # Use v2 API explicitly
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    metadata = load_patient_metadata(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating zoomed frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create zoomed plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_zoomed_timeline_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, window_hours, metadata, frame_path
        )
        frame_paths.append(str(frame_path))
    
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions_zoomed.mp4"
    
    # Create video with imageio (v2 API, fixed dimensions)
    writer = iio.get_writer(output_path, fps=fps, codec='libx264', quality=8, 
                           macro_block_size=1)
    
    for frame_path in frame_paths:
        frame = iio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"Zoomed video created successfully: {output_path}")


def create_zoomed_timeline_frame(icustay_id, treat, map_df, pos, pred, 
                                 intime, outtime, current_time, window_hours, metadata, save_path):
    """
    Create a zoomed-in timeline visualization with a rolling window.
    
    Parameters:
    -----------
    window_hours : float
        Size of the window in hours (e.g., 6 shows ±3 hours from current time)
    metadata : pd.DataFrame
        Patient metadata including demographics and comorbidities
    """
    fig = plt.figure(figsize=(20, 11), constrained_layout=True)
    
    # Create grid: 1 row for metadata panel, 4 rows for timeline
    #gs = fig.add_gridspec(5, 1, height_ratios=[0.8, 1, 1, 1, 1], hspace=0.15)
    gs = fig.add_gridspec(
    5, 1,
    height_ratios=[1.3, 1, 1, 1, 1]
)
    # Metadata panel at top
    ax_meta = fig.add_subplot(gs[0, 0])
    
    # Timeline panels
    axes = [fig.add_subplot(gs[i+1, 0]) for i in range(4)]
    
    # Add patient metadata panel
    add_patient_metadata_panel(ax_meta, metadata, icustay_id)
    
    fig.suptitle(f'ICU Stay {icustay_id} - Time: {current_time.strftime("%Y-%m-%d %H:%M")} [Zoomed View: ±{window_hours/2:.1f}h]', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Convert times to hours from admission for easier plotting
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # Calculate window bounds
    window_start = max(0, current_hour - window_hours / 2)
    window_end = min(total_hours, current_hour + window_hours / 2)
    
    # Adjust if we're at the boundaries
    if current_hour < window_hours / 2:
        window_end = min(total_hours, window_hours)
    if current_hour > total_hours - window_hours / 2:
        window_start = max(0, total_hours - window_hours)
    
    # ============== Panel 1: MAP Values ==============
    ax1 = axes[0]
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        values = map_df['valuenum'].values
        
        # Filter to window
        window_mask = [(h >= window_start and h <= window_end) for h in hours]
        window_hours_list = [h for h, m in zip(hours, window_mask) if m]
        window_values = [v for v, m in zip(values, window_mask) if m]
        
        if window_hours_list:
            ax1.plot(window_hours_list, window_values, 'k-', linewidth=2, marker='o', markersize=5)
        
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Hypotension Threshold (65mmHg)')
    
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8, label='Current time')
    ax1.set_ylabel('MAP (mmHg)', fontsize=14, fontweight='bold')
    ax1.set_title('Context: Mean Arterial Pressure', fontsize=12, loc='left', fontweight='bold')
    ax1.grid(True, alpha=0.4, linewidth=0.5)
    ax1.legend(loc='upper right', fontsize=11)
    ax1.set_ylim(40, 120)
    
    # ============== Panel 2: Ground Truth Positive Events ==============
    ax2 = axes[1]
    ax2.set_ylabel('Ground Truth', fontsize=14, fontweight='bold')
    ax2.set_title('True Catecholamine Initiation Windows', fontsize=12, loc='left', fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        
        # Only show if visible in window and has started
        if start_hour <= current_hour and end_hour >= window_start and start_hour <= window_end:
            width = end_hour - start_hour
            
            # Color based on whether event is in past or ongoing
            if end_hour <= current_hour:
                color = 'darkred'
                alpha = 0.6
                label = 'Past Event'
            else:
                color = 'red'
                alpha = 0.9
                label = 'Ongoing Event'
            
            rect = Rectangle((start_hour, 0.3), width, 1.4, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=2)
            ax2.add_patch(rect)
            
            # Add label at start of event
            if start_hour >= window_start:
                ax2.text(start_hour, 1.0, '▼', ha='center', va='center', 
                        fontsize=16, fontweight='bold', color='darkred')
    
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax2.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # ============== Panel 3: Model Predictions ==============
    ax3 = axes[2]
    ax3.set_ylabel('Predictions', fontsize=14, fontweight='bold')
    ax3.set_title('Model Predicted Risk Windows', fontsize=12, loc='left', fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            
            # Only show if visible in window
            if end_hour >= window_start and start_hour <= window_end:
                width = end_hour - start_hour
                
                # Predictions fade based on when they were made
                pred_time = to_hours(p['context_end'])
                hours_old = current_hour - pred_time
                alpha = max(0.4, 1.0 - (hours_old / 12))  # Fade over 12 hours
                
                rect = Rectangle((start_hour, 0.3), width, 1.4,
                               facecolor='orange', edgecolor='black', alpha=alpha, linewidth=2)
                ax3.add_patch(rect)
                
                # Add marker at prediction time
                if pred_time >= window_start and pred_time <= window_end:
                    ax3.plot(pred_time, 1.0, 'o', color='darkorange', markersize=8, 
                            markeredgecolor='black', markeredgewidth=1.5, zorder=10)
    
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax3.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # ============== Panel 4: Treatment Starts ==============
    ax4 = axes[3]
    ax4.set_ylabel('Treatments', fontsize=14, fontweight='bold')
    ax4.set_title('Vasopressor Treatment Initiations', fontsize=12, loc='left', fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    ax4.set_xlabel('Hours from ICU Admission', fontsize=14, fontweight='bold')
    
    # Show treatment starts as vertical lines
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        
        # Only show if in window
        if start_hour >= window_start and start_hour <= window_end:
            ax4.axvline(x=start_hour, color='green', linewidth=4, alpha=0.8, zorder=5)
            ax4.plot(start_hour, 1.0, 'v', color='darkgreen', markersize=12, 
                    markeredgecolor='black', markeredgewidth=1.5, zorder=10)
            ax4.text(start_hour, 0.3, t['drug_label'], rotation=45, 
                    va='top', ha='right', fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))
    
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax4.grid(True, alpha=0.4, axis='x', linewidth=0.5)
    
    # Set x-axis limits for all panels (zoomed window)
    for ax in axes:
        ax.set_xlim(window_start, window_end)
        # Add time markers every hour
        ax.set_xticks(np.arange(np.ceil(window_start), np.floor(window_end) + 1, 1))
    
    # Add minor gridlines every 15 minutes
    for ax in axes:
        ax.set_xticks(np.arange(window_start, window_end, 0.25), minor=True)
        ax.grid(True, which='minor', alpha=0.2, linewidth=0.3)
    
    #plt.tight_layout(rect=[0, 0, 1, 0.99])  # Leave space for suptitle
    # Use fixed bbox instead of 'tight' to ensure consistent dimensions
    plt.savefig(save_path, dpi=120, bbox_inches=None)
    plt.close()


def add_patient_metadata_panel(ax, metadata, icustay_id):
    """
    Add a compact patient metadata panel showing demographics and comorbidities.
    Uses color-coded boxes for comorbidities (present/absent).
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    if metadata.empty:
        ax.text(0.5, 0.5, 'No metadata available', ha='center', va='center', fontsize=12)
        return
    
    patient = metadata.iloc[0]
    
    # Background box
    bg_rect = Rectangle((0.005, 0.05), 0.99, 0.9, facecolor='#f8f9fa', 
                        edgecolor='#dee2e6', linewidth=2, zorder=0)
    ax.add_patch(bg_rect)
    
    # Left side: Demographics
    y_pos = 0.75
    demographics_x = 0.02
    
    # Format demographics
    gender = patient.get('gender', 'Unknown')
    ethnicity = patient.get('ethnicity_group', 'Unknown')
    age = patient.get('age', 'Unknown')
    height = patient.get('height', None)
    weight = patient.get('weight', None)
    
    # Calculate BMI if available
    bmi_text = ""
    if height and weight and height > 0:
        bmi = weight / ((height / 100) ** 2)
        bmi_text = f" (BMI: {bmi:.1f})"
    
    demo_text = (
        f"Gender: {gender}  |  Age: {age} yrs  |  Ethnicity: {ethnicity}\n"
        f"Height: {height if height else 'N/A'} cm  |  Weight: {weight if weight else 'N/A'} kg{bmi_text}"
    )
    
    ax.text(demographics_x, y_pos, demo_text, fontsize=10, va='top', 
            fontweight='normal', family='monospace')
    
    # Right side: Comorbidities with color-coded indicators
    comorb_x_start = 0.02
    comorb_y = 0.35
    
    ax.text(comorb_x_start, comorb_y + 0.12, 'Comorbidities:', 
            fontsize=11, fontweight='bold', va='top')
    
    # Define comorbidities to display
    comorbidities = [
        ('obesity', 'Obesity'),
        ('hypertension', 'Hypertension'),
        ('diabetes', 'Diabetes'),
        ('kidney_disease', 'Kidney Disease'),
        ('lung_disease', 'Lung Disease'),
        ('heart_disease', 'Heart Disease'),
    ]
    
    # Create color-coded boxes for each comorbidity
    box_width = 0.15
    box_height = 0.08
    boxes_per_row = 3
    x_spacing = 0.16
    y_spacing = 0.11
    
    for idx, (col_name, display_name) in enumerate(comorbidities):
        row = idx // boxes_per_row
        col = idx % boxes_per_row
        
        box_x = comorb_x_start + col * x_spacing
        box_y = comorb_y - row * y_spacing
        
        # Check if comorbidity is present
        is_present = patient.get(col_name, 0) == 1
        
        # Color scheme
        if is_present:
            box_color = '#ffcccc'  # Light red
            edge_color = '#cc0000'  # Dark red
            text_color = '#cc0000'
            text_weight = 'bold'
            symbol = '✓'
        else:
            box_color = '#e8f4f8'  # Light blue-gray
            edge_color = '#b0bec5'  # Gray
            text_color = '#666666'
            text_weight = 'normal'
            symbol = '−'
        
        # Draw box
        rect = Rectangle((box_x, box_y - box_height), box_width, box_height,
                        facecolor=box_color, edgecolor=edge_color, 
                        linewidth=2, zorder=1)
        ax.add_patch(rect)
        
        # Add text
        ax.text(box_x + box_width/2, box_y - box_height/2, 
               f"{symbol} {display_name}", 
               ha='center', va='center', fontsize=9, 
               color=text_color, fontweight=text_weight, zorder=2)
    
    # Add legend/key at bottom right
    legend_x = 0.85
    legend_y = 0.15
    
    # Present indicator
    small_box = Rectangle((legend_x, legend_y), 0.02, 0.05, 
                          facecolor='#ffcccc', edgecolor='#cc0000', linewidth=1.5)
    ax.add_patch(small_box)
    ax.text(legend_x + 0.025, legend_y + 0.025, 'Present', 
           va='center', fontsize=8, color='#666666')
    
    # Absent indicator
    small_box2 = Rectangle((legend_x, legend_y - 0.08), 0.02, 0.05, 
                           facecolor='#e8f4f8', edgecolor='#b0bec5', linewidth=1.5)
    ax.add_patch(small_box2)
    ax.text(legend_x + 0.025, legend_y - 0.055, 'Absent', 
           va='center', fontsize=8, color='#666666')


def create_dual_view_video(icustay_id, interval_minutes=15, frames_around_event=10, 
                           window_hours=6, output_path=None, fps=5):
    """
    Create a video with both full timeline and zoomed view side by side.
    
    Top: Full timeline overview
    Bottom: Zoomed rolling window
    """
    import imageio.v2 as iio  # Use v2 API explicitly
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty or pos.empty:
        print(f"Insufficient data for ICU stay {icustay_id}")
        return
    
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames with dual view")
    
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}")
        
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_dual_view_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, window_hours, frame_path
        )
        frame_paths.append(str(frame_path))
    
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions_dual.mp4"
    
    writer = iio.get_writer(output_path, fps=fps, codec='libx264', quality=8,
                           macro_block_size=1)
    
    for frame_path in frame_paths:
        frame = iio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"Dual-view video created: {output_path}")


def create_dual_view_frame(icustay_id, treat, map_df, pos, pred, 
                           intime, outtime, current_time, window_hours, save_path):
    """Create a frame with both overview and zoomed view."""
    
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(8, 1, hspace=0.3)
    
    # Top section: Full overview (4 rows)
    ax_overview = [fig.add_subplot(gs[i, 0]) for i in range(4)]
    
    # Bottom section: Zoomed view (4 rows)
    ax_zoom = [fig.add_subplot(gs[i+4, 0]) for i in range(4)]
    
    fig.suptitle(f'ICU Stay {icustay_id} - Time: {current_time.strftime("%Y-%m-%d %H:%M")}', 
                 fontsize=18, fontweight='bold')
    
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # Plot full overview
    plot_overview_section(ax_overview, map_df, pos, pred, treat, intime, current_time, 
                         current_hour, total_hours, to_hours, "OVERVIEW")
    
    # Plot zoomed section
    plot_zoomed_section(ax_zoom, map_df, pos, pred, treat, intime, outtime, current_time, 
                       current_hour, total_hours, window_hours, to_hours, "ZOOMED VIEW")
    
    plt.savefig(save_path, dpi=100, bbox_inches=None)
    plt.close()


def plot_overview_section(axes, map_df, pos, pred, treat, intime, current_time, 
                          current_hour, total_hours, to_hours, title_prefix):
    """Helper to plot full overview."""
    # Similar to create_timeline_frame but with title prefix
    ax1, ax2, ax3, ax4 = axes
    
    # MAP
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        ax1.plot(hours, map_df['valuenum'], 'k-', linewidth=0.5, marker='o', markersize=2)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.3)
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax1.set_ylabel('MAP', fontsize=10, fontweight='bold')
    ax1.set_title(f'{title_prefix} - Mean Arterial Pressure', fontsize=10, loc='left')
    ax1.set_ylim(40, 120)
    ax1.grid(True, alpha=0.2)
    
    # Ground truth
    ax2.set_ylabel('Truth', fontsize=10, fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        if start_hour <= current_hour:
            color = 'darkred' if end_hour <= current_hour else 'red'
            alpha = 0.5 if end_hour <= current_hour else 0.8
            rect = Rectangle((start_hour, 0.5), end_hour - start_hour, 1.0, 
                           facecolor=color, alpha=alpha)
            ax2.add_patch(rect)
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax2.grid(True, alpha=0.2, axis='x')
    
    # Predictions
    ax3.set_ylabel('Pred', fontsize=10, fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            pred_time = to_hours(p['context_end'])
            hours_old = current_hour - pred_time
            alpha = max(0.3, 1.0 - (hours_old / 24))
            rect = Rectangle((start_hour, 0.5), end_hour - start_hour, 1.0,
                           facecolor='orange', alpha=alpha)
            ax3.add_patch(rect)
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax3.grid(True, alpha=0.2, axis='x')
    
    # Treatments
    ax4.set_ylabel('Treat', fontsize=10, fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        ax4.axvline(x=start_hour, color='green', linewidth=2, alpha=0.7)
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax4.grid(True, alpha=0.2, axis='x')
    ax4.set_xlabel('Hours from ICU Admission', fontsize=10)
    
    for ax in axes:
        ax.set_xlim(0, total_hours)


def plot_zoomed_section(axes, map_df, pos, pred, treat, intime, outtime, current_time, 
                       current_hour, total_hours, window_hours, to_hours, title_prefix):
    """Helper to plot zoomed view."""
    # Calculate window
    window_start = max(0, current_hour - window_hours / 2)
    window_end = min(total_hours, current_hour + window_hours / 2)
    
    if current_hour < window_hours / 2:
        window_end = min(total_hours, window_hours)
    if current_hour > total_hours - window_hours / 2:
        window_start = max(0, total_hours - window_hours)
    
    ax1, ax2, ax3, ax4 = axes
    
    # MAP (zoomed)
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        values = map_df['valuenum'].values
        window_mask = [(h >= window_start and h <= window_end) for h in hours]
        window_hours_list = [h for h, m in zip(hours, window_mask) if m]
        window_values = [v for v, m in zip(values, window_mask) if m]
        if window_hours_list:
            ax1.plot(window_hours_list, window_values, 'k-', linewidth=2, marker='o', markersize=5)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.5, linewidth=2)
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax1.set_ylabel('MAP', fontsize=12, fontweight='bold')
    ax1.set_title(f'{title_prefix} (±{window_hours/2:.1f}h) - MAP', fontsize=11, loc='left', fontweight='bold')
    ax1.set_ylim(40, 120)
    ax1.grid(True, alpha=0.4)
    
    # Ground truth (zoomed)
    ax2.set_ylabel('Truth', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        if start_hour <= current_hour and end_hour >= window_start and start_hour <= window_end:
            color = 'darkred' if end_hour <= current_hour else 'red'
            alpha = 0.6 if end_hour <= current_hour else 0.9
            rect = Rectangle((start_hour, 0.3), end_hour - start_hour, 1.4, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=2)
            ax2.add_patch(rect)
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax2.grid(True, alpha=0.4, axis='x')
    
    # Predictions (zoomed)
    ax3.set_ylabel('Pred', fontsize=12, fontweight='bold')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            if end_hour >= window_start and start_hour <= window_end:
                pred_time = to_hours(p['context_end'])
                hours_old = current_hour - pred_time
                alpha = max(0.4, 1.0 - (hours_old / 12))
                rect = Rectangle((start_hour, 0.3), end_hour - start_hour, 1.4,
                               facecolor='orange', edgecolor='black', alpha=alpha, linewidth=2)
                ax3.add_patch(rect)
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax3.grid(True, alpha=0.4, axis='x')
    
    # Treatments (zoomed)
    ax4.set_ylabel('Treat', fontsize=12, fontweight='bold')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        if start_hour >= window_start and start_hour <= window_end:
            ax4.axvline(x=start_hour, color='green', linewidth=4, alpha=0.8)
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=3, alpha=0.8)
    ax4.grid(True, alpha=0.4, axis='x')
    ax4.set_xlabel('Hours from ICU Admission', fontsize=12, fontweight='bold')
    
    for ax in axes:
        ax.set_xlim(window_start, window_end)

def load_patient_metadata(icustay_id):
    """Load patient demographics and comorbidities."""
    df = pd.read_sql("""
        SELECT gender, ethnicity_group, age, height, weight, 
               obesity, hypertension, diabetes, kidney_disease, 
               lung_disease, heart_disease
        FROM ce_approach.mv_metadata
        WHERE icustay_id = %(icustay_id)s
    """, engine, params={"icustay_id": icustay_id})
    return df


# Example usage:
create_zoomed_prediction_video(icustay_id=202300, interval_minutes=15, frames_around_event=10, window_hours=6)
# create_dual_view_video(icustay_id=229240, interval_minutes=15, frames_around_event=10, window_hours=6)

Generated 29 frames focused on 1 ground truth events
Time range: 2192-06-20 18:14:22 to 2192-06-21 01:14:22
Generating zoomed frames...
Frame 1/29: 2192-06-20 18:14:22
Frame 11/29: 2192-06-20 20:44:22
Frame 21/29: 2192-06-20 23:14:22
Zoomed video created successfully: /dss/work/rirg2545/actionable-hypotension/simulation/202300_predictions_zoomed.mp4


In [27]:
create_zoomed_prediction_video(icustay_id=200952, interval_minutes=15, frames_around_event=10, window_hours=6)

Generated 136 frames focused on 7 ground truth events
Time range: 2139-09-23 12:38:34 to 2139-09-28 04:08:34
Generating zoomed frames...
Frame 1/136: 2139-09-23 12:38:34
Frame 11/136: 2139-09-23 15:08:34
Frame 21/136: 2139-09-23 17:38:34
Frame 31/136: 2139-09-23 22:53:34
Frame 41/136: 2139-09-24 01:23:34
Frame 51/136: 2139-09-25 08:53:34
Frame 61/136: 2139-09-25 11:23:34
Frame 71/136: 2139-09-25 13:53:34
Frame 81/136: 2139-09-26 07:08:34
Frame 91/136: 2139-09-26 09:38:34
Frame 101/136: 2139-09-26 21:23:34
Frame 111/136: 2139-09-26 23:53:34
Frame 121/136: 2139-09-28 00:23:34
Frame 131/136: 2139-09-28 02:53:34
Zoomed video created successfully: /dss/work/rirg2545/actionable-hypotension/simulation/200952_predictions_zoomed.mp4


In [16]:
#create_prediction_gif(icustay_id=220527, interval_minutes=15)
id_list = [
200758,
200952,
201098,
201117,
201145,
201299,
201430,
201624,
201834,
202256,
202300
]
for id in id_list:
    create_prediction_video(icustay_id=id, interval_minutes=15)


Generated 21 frames focused on 1 ground truth events
Time range: 2189-06-02 13:56:52 to 2189-06-02 18:56:52
Generating frames...
Frame 1/21: 2189-06-02 13:56:52
Frame 11/21: 2189-06-02 16:26:52
Frame 21/21: 2189-06-02 18:56:52


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Generated 136 frames focused on 7 ground truth events
Time range: 2139-09-23 12:38:34 to 2139-09-28 04:08:34
Generating frames...
Frame 1/136: 2139-09-23 12:38:34
Frame 11/136: 2139-09-23 15:08:34
Frame 21/136: 2139-09-23 17:38:34
Frame 31/136: 2139-09-23 22:53:34
Frame 41/136: 2139-09-24 01:23:34
Frame 51/136: 2139-09-25 08:53:34
Frame 61/136: 2139-09-25 11:23:34
Frame 71/136: 2139-09-25 13:53:34
Frame 81/136: 2139-09-26 07:08:34
Frame 91/136: 2139-09-26 09:38:34
Frame 101/136: 2139-09-26 21:23:34
Frame 111/136: 2139-09-26 23:53:34
Frame 121/136: 2139-09-28 00:23:34
Frame 131/136: 2139-09-28 02:53:34


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Generated 37 frames focused on 4 ground truth events
Time range: 2124-12-23 05:54:25 to 2124-12-23 14:54:25
Generating frames...
Frame 1/37: 2124-12-23 05:54:25
Frame 11/37: 2124-12-23 08:24:25
Frame 21/37: 2124-12-23 10:54:25
Frame 31/37: 2124-12-23 13:24:25


...

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Generated 39 frames focused on 2 ground truth events
Time range: 2120-09-14 10:04:33 to 2120-09-14 19:34:33
Generating frames...
Frame 1/39: 2120-09-14 10:04:33
Frame 11/39: 2120-09-14 12:34:33
Frame 21/39: 2120-09-14 15:04:33
Frame 31/39: 2120-09-14 17:34:33


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Generated 19 frames focused on 1 ground truth events
Time range: 2192-06-20 18:14:22 to 2192-06-20 22:44:22
Generating frames...
Frame 1/19: 2192-06-20 18:14:22
Frame 11/19: 2192-06-20 20:44:22


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
